# **Project Name**    -



##### **Project Type**    - Classification
##### **Contribution**    - Individual


# **Project Summary -**

This project focuses on analyzing and improving customer satisfaction at Flipkart, one of India's largest e-commerce platforms. It uses a real customer service dataset containing around 22,430 support interactions collected across multiple channels (Email, Inbound calls, Outcall), covering details like issue category, response times, customer remarks, agent information, and a Customer Satisfaction (CSAT) score for each interaction.

The core goal of the project is twofold: first, to understand what factors actually drive customer satisfaction or dissatisfaction in Flipkart's support operations — such as how quickly issues are resolved, which channels or issue types perform worse, and how agent experience or supervisor performance affects outcomes. Second, to build a machine learning model that can predict whether a customer is likely to be satisfied or dissatisfied based on these interaction details, so that Flipkart can proactively identify and address problems before they escalate into churn or negative reviews.

To achieve this, the project involves cleaning and preparing messy real-world data, engineering meaningful features (like response time and sentiment from customer feedback text), exploring the data visually to uncover patterns, statistically testing key assumptions, and finally training and comparing multiple classification models to find the one best suited to catching at-risk customers — not just the one with the highest raw accuracy.

Ultimately, this is a customer experience analytics project: it turns raw support-ticket data into clear, actionable business insights and a working prediction model, helping Flipkart make targeted improvements to its customer service strategy, boost satisfaction scores, and improve long-term customer retention and loyalty.

# **GitHub Link -**

https://github.com/arvindkumar111/Classification---Flipkart-Customer-Service-Satisfaction




# **Problem Statement**


Company: Flipkart (major e-commerce platform)

Problem/Context: In a highly competitive e-commerce market, strong customer service is a key differentiator for retaining customers and sustaining growth. Flipkart wants to better understand and improve customer satisfaction across its support operations.

Data: Customer interactions, feedback, and satisfaction scores collected across various customer support channels.

Objectives:

Identify the key drivers behind customer satisfaction (CSAT)
Evaluate performance across different customer service teams/agents
Develop data-driven strategies to improve the overall support experience

Expected Outcomes:

Faster resolution of customer issues
Support strategies tailored to diverse customer needs/expectations
Optimized performance of service agents
Improved CSAT and other satisfaction metrics
Increased brand loyalty and customer retention

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


### Dataset Loading

In [ ]:
# Load Dataset
df=pd.read_csv("/content/Customer_support_data (1).csv")

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
df.shape

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df.isnull().mean()*100

In [ ]:
# Visualizing the missing values


### What did you know about your dataset?

Answer Here

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe
df.describe()

### Variables Description

Answer Here

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
# Check cardinality (unique value counts) of key categorical columns
cat_cols = ['channel_name', 'category', 'Sub-category', 'Agent_name',
            'Supervisor', 'Manager', 'Tenure Bucket', 'Agent Shift', 'Product_category']

for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")

# Check CSAT Score distribution (this is our TARGET — very important)
print("\nCSAT Score distribution:")
print(df['CSAT Score'].value_counts(normalize=True).sort_index())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
# Make a working copy so original stays untouched
data = df.copy()

# --- 1. Drop columns that are unusable or not useful for modeling ---
# connected_handling_time: 99.7% missing -> unusable
# Unique id, Order_id: identifiers, no predictive value
# order_date_time, Customer_City, Item_price, Product_category: ~83% missing
#   -> we'll engineer a "has_order_info" flag instead of using raw columns
data['has_order_info'] = data['order_date_time'].notnull().astype(int)

cols_to_drop = ['connected_handling_time', 'Unique id', 'Order_id',
                'order_date_time', 'Customer_City', 'Item_price', 'Product_category']
data = data.drop(columns=cols_to_drop)

# --- 2. Create the target variable: Satisfied (1) vs Not Satisfied (0) ---
data['Satisfied'] = data['CSAT Score'].apply(lambda x: 1 if x >= 4 else 0)

print(data['Satisfied'].value_counts(normalize=True))

In [ ]:
# --- 1. Parse datetime columns with explicit formats (avoids ambiguity/errors) ---
data['Issue_reported at'] = pd.to_datetime(data['Issue_reported at'],
                                             format='%d/%m/%Y %H:%M')
data['issue_responded'] = pd.to_datetime(data['issue_responded'],
                                           format='%d/%m/%Y %H:%M')
data['Survey_response_Date'] = pd.to_datetime(data['Survey_response_Date'],
                                                format='%d-%b-%y')

# --- 2. Engineer Response Time (in minutes) ---
# This is likely one of our strongest predictors: how fast was the issue handled?
data['response_time_min'] = (data['issue_responded'] - data['Issue_reported at']).dt.total_seconds() / 60

# Sanity check: response time should never be negative
print("Negative response times:", (data['response_time_min'] < 0).sum())
print(data['response_time_min'].describe())

In [ ]:
# --- 3. Extract useful time-based features ---
# Hour of day and day of week the issue was reported — captures patterns like
# "night-time issues get slower responses -> lower CSAT"
data['report_hour'] = data['Issue_reported at'].dt.hour
data['report_dayofweek'] = data['Issue_reported at'].dt.dayofweek  # 0=Monday

# --- 4. Handle any missing/invalid response times ---
# If response_time_min is negative or NaN, treat as missing and impute with median
data.loc[data['response_time_min'] < 0, 'response_time_min'] = np.nan
median_response = data['response_time_min'].median()
data['response_time_min'] = data['response_time_min'].fillna(median_response)

print("\nMissing response_time_min after cleaning:", data['response_time_min'].isnull().sum())

In [ ]:
# --- 1. Cap extreme outliers at 99th percentile (winsorizing) ---
cap_value = data['response_time_min'].quantile(0.99)
print("99th percentile cap value:", cap_value)

data['response_time_min_capped'] = np.where(
    data['response_time_min'] > cap_value, cap_value, data['response_time_min']
)

# --- 2. Log transform (log1p handles zero values safely, since log(0) is undefined) ---
data['response_time_log'] = np.log1p(data['response_time_min_capped'])

# Quick check of the new distribution
print(data[['response_time_min', 'response_time_min_capped', 'response_time_log']].describe())

In [ ]:
# --- 1. Flag whether a customer left a remark ---
data['has_remark'] = data['Customer Remarks'].notnull().astype(int)

# --- 2. Fill missing remarks with empty string so text processing doesn't break ---
data['Customer Remarks'] = data['Customer Remarks'].fillna('')

# --- 3. Simple, interpretable sentiment scoring using TextBlob ---
# TextBlob gives a polarity score from -1 (very negative) to +1 (very positive)
from textblob import TextBlob

def get_sentiment(text):
    if text.strip() == '':
        return 0.0  # neutral score for no remark
    return TextBlob(text).sentiment.polarity

data['remark_sentiment'] = data['Customer Remarks'].apply(get_sentiment)

print(data[['has_remark', 'remark_sentiment']].describe())
print(data[data['has_remark']==1][['Customer Remarks','remark_sentiment']].head(10))

In [ ]:
# --- 4. Optional: keyword flags for common complaint themes ---
# These are common, high-signal complaint words in e-commerce support text
keywords = ['refund', 'delay', 'broken', 'damaged', 'worst', 'poor', 'rude', 'wrong']

for word in keywords:
    data[f'kw_{word}'] = data['Customer Remarks'].str.lower().str.contains(word).astype(int)

print(data[[f'kw_{w}' for w in keywords]].sum())

In [ ]:
# --- Low cardinality columns: one-hot encode safely ---
low_card_cols = ['channel_name', 'category', 'Tenure Bucket', 'Agent Shift']

data_encoded = pd.get_dummies(data, columns=low_card_cols, drop_first=True)

print(data_encoded.shape)

In [ ]:
# --- Medium cardinality columns: frequency encoding ---
medium_card_cols = ['Sub-category', 'Supervisor']

for col in medium_card_cols:
    freq_map = data_encoded[col].value_counts(normalize=True)
    data_encoded[col + '_freq'] = data_encoded[col].map(freq_map)

# Drop original medium-cardinality text columns after encoding
data_encoded = data_encoded.drop(columns=medium_card_cols)

print(data_encoded.shape)
data_encoded.head()

In [ ]:
# --- Merge broken + damaged into one signal (semantically similar, both individually sparse) ---
data_encoded['kw_broken_or_damaged'] = ((data_encoded['kw_broken'] == 1) | (data_encoded['kw_damaged'] == 1)).astype(int)
data_encoded = data_encoded.drop(columns=['kw_broken', 'kw_damaged'])

print(data_encoded['kw_broken_or_damaged'].sum())

In [ ]:
# --- Drop columns no longer needed for modeling ---
# Raw datetime columns: already extracted useful features (response_time, hour, dayofweek)
# Customer Remarks (raw text): already extracted sentiment + keywords
# Agent_name: too high cardinality
# Manager: redundant with Supervisor
# CSAT Score: this is the source of our target 'Satisfied' - must drop to avoid leakage!
cols_to_drop_final = ['Issue_reported at', 'issue_responded', 'Survey_response_Date',
                       'Customer Remarks', 'Agent_name', 'Manager', 'CSAT Score']

model_df = data_encoded.drop(columns=cols_to_drop_final)

print("Final shape:", model_df.shape)
print("\nFinal columns:\n", model_df.columns.tolist())
print("\nAny remaining nulls?\n", model_df.isnull().sum().sum())

In [ ]:
# --- Find exactly which columns and rows have nulls ---
null_cols = model_df.columns[model_df.isnull().any()]
print("Columns with nulls:\n", model_df[null_cols].isnull().sum())

# Show the actual rows with nulls
print("\nRows with nulls:")
print(model_df[model_df.isnull().any(axis=1)])

In [ ]:
# --- Drop the single row with cascading nulls (incomplete source data, not worth imputing) ---
model_df = model_df.dropna()

print("Final shape after dropping incomplete row:", model_df.shape)
print("Remaining nulls:", model_df.isnull().sum().sum())

### What all manipulations have you done and insights you found?

## What all manipulations have you done and insights you found?

**Cleaning:** Dropped `connected_handling_time` (99.7% missing), ID columns (`Unique id`, `Order_id`), and columns with ~83% missing (`order_date_time`, `Customer_City`, `Item_price`, `Product_category`) — but kept a `has_order_info` flag from them. Dropped `Agent_name` (1246 unique values, too sparse) and `Manager` (redundant with `Supervisor`). No duplicate rows found.

**Target:** `CSAT Score` was heavily imbalanced (68.5% scored 5, 14% scored 1). Reframed as binary `Satisfied` (CSAT 4–5) vs `Not Satisfied` (CSAT 1–3) → 81.2% / 18.8% split. Dropped raw `CSAT Score` to avoid target leakage.

**Datetime features:** Parsed timestamps and engineered `response_time_min`. Found strong right-skew (median 6 min, mean 169.6 min, max ~4 days) — capped at 99th percentile and added a log-transformed version. Also extracted `report_hour` and `report_dayofweek`.

**Text features (`Customer Remarks`):** Only 33% of customers left a remark (`has_remark` flag added). Applied TextBlob sentiment scoring (neutral-filled for missing remarks) and added keyword flags for common complaint terms (`refund`, `worst`, `poor`, etc.). Merged rare `broken`/`damaged` keywords into one flag due to low individual counts.

**Encoding:** One-hot encoded low-cardinality columns (`channel_name`, `category`, `Tenure Bucket`, `Agent Shift`). Frequency-encoded medium-cardinality columns (`Sub-category`, `Supervisor`) to avoid high dimensionality and leakage.

**Final check:** Dropped 1 row with cascading nulls (incomplete source data). Final dataset: **22,429 rows × 39 columns**, fully numeric, no missing values, leakage-free, ready for modeling.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart 1: CSAT Score Distribution
plt.figure(figsize=(7,5))
sns.countplot(data=df, x='CSAT Score', order=sorted(df['CSAT Score'].dropna().unique()), palette='viridis')
plt.title('Distribution of CSAT Scores')
plt.xlabel('CSAT Score')
plt.ylabel('Count')
plt.show()

##### 1. Why did you pick the specific chart?

 A count plot suits a discrete, ordinal target — it shows category frequencies clearly, ideal for spotting class imbalance.



##### 2. What is/are the insight(s) found from the chart?

CSAT is heavily skewed — 68.5% of scores are 5, 14% are 1, and scores 2–3 combined are under 5%. Customers are mostly polarized: very happy or very unhappy.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — confirms most customers are satisfied, and highlights that the ~14-19% dissatisfied segment is a clear, targetable group for improvement rather than a broad, spread-out problem.

Any negative-growth insight? Yes — the near-total absence of "neutral" (2–3) scores suggests customers rarely give lukewarm feedback; dissatisfaction tends to be strong (score 1). This means unresolved issues likely drive strongly negative word-of-mouth/reviews rather than mild complaints, so dissatisfaction here carries outsized reputational risk.

#### Chart - 2

In [ ]:
# Chart 2: Satisfaction Rate by Channel
plt.figure(figsize=(7,5))
sns.barplot(data=data, x='channel_name', y='Satisfied', palette='mako', ci=None)
plt.title('Satisfaction Rate by Channel')
plt.ylabel('Satisfaction Rate')
plt.xlabel('Channel')
plt.show()


##### 1. Why did you pick the specific chart?

A bar plot is ideal for comparing a continuous metric (satisfaction rate) across a small number of categories (3 channels) — quick visual comparison of averages.

##### 2. What is/are the insight(s) found from the chart?

Email has a noticeably lower satisfaction rate (70.5%) compared to Inbound (81.5%) and Outcall (81.4%), which are nearly identical.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — this pinpoints Email support as an underperforming channel, giving Flipkart a clear, specific area to investigate (e.g., slower response times, less personal interaction) and improve.

Any negative-growth insight? Yes — since Email is likely used for more detailed/written complaints, its lower satisfaction combined with being a "written record" channel means dissatisfied experiences here are more likely to be escalated, screenshotted, or shared publicly (e.g., social media complaints) compared to a phone call.

#### Chart - 3

In [ ]:
# Chart 3: Satisfaction Rate by Category
plt.figure(figsize=(10,6))
cat_satisfaction = data.groupby('category')['Satisfied'].mean().sort_values()
sns.barplot(x=cat_satisfaction.values, y=cat_satisfaction.index, palette='mako')
plt.title('Satisfaction Rate by Issue Category')
plt.xlabel('Satisfaction Rate')
plt.ylabel('Category')
plt.show()


##### 1. Why did you pick the specific chart?

Horizontal bar chart handles 12 categories cleanly, sorted by value so the best/worst performers are immediately visible without rotated labels.

##### 2. What is/are the insight(s) found from the chart?

: "Others" (60%), "Cancellation" (72.3%), and "Order Related" (75.7%) have the lowest satisfaction, while "App/website" (91.7%), "Onboarding related" (90%), and "Offers & Cashback" (89%) have the highest.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — clearly ranks issue types by satisfaction, letting Flipkart prioritize process improvements for Cancellation and Order Related handling specifically, rather than spreading effort evenly across all categories.

**Any negative-growth insight?** Yes — "Order Related" and "Cancellation" are core transactional issues (not minor queries), and they show the weakest satisfaction. Since these touch the actual purchase experience directly, poor handling here risks losing repeat customers, not just one-off complaint frustration — a more direct threat to retention than, say, a website UX complaint.

#### Chart - 4

In [ ]:
# Chart 4: Response Time vs Satisfaction

plt.figure(figsize=(8,6))
sns.boxplot(data=data, x='Satisfied', y='response_time_min_capped', palette='coolwarm')
plt.title('Response Time by Satisfaction Status')
plt.xlabel('Satisfied (0=No, 1=Yes)')
plt.ylabel('Response Time (minutes, capped)')
plt.show()


##### 1. Why did you pick the specific chart?

A box plot suits continuous, skewed data — it shows the median and spread by group, which a simple average would misrepresent given the skew we found earlier.

##### 2. What is/are the insight(s) found from the chart?

Dissatisfied customers have a median response time of 20 minutes vs. 5 minutes for satisfied customers (4x slower) — and the gap is even larger in the mean (296 vs 126 minutes), confirming slow response strongly correlates with dissatisfaction.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — this is one of the clearest, most actionable levers: reducing response time directly targets a measurable driver of dissatisfaction, and Flipkart can set concrete SLA targets (e.g., "respond within 5-10 minutes") to improve CSAT.

**Any negative-growth insight?** Yes — the large mean-vs-median gap among dissatisfied customers (296 vs 20 min median) shows a subset of customers face extremely long delays. These extreme cases are likely to generate the harshest complaints/reviews, meaning a small number of severely mishandled tickets could disproportionately damage brand reputation.

#### Chart - 5

In [ ]:
# Chart 5: Satisfaction Rate by Agent Shift

plt.figure(figsize=(7,5))
shift_satisfaction = data.groupby('Agent Shift')['Satisfied'].mean().sort_values()
sns.barplot(x=shift_satisfaction.index, y=shift_satisfaction.values, palette='mako')
plt.title('Satisfaction Rate by Agent Shift')
plt.xlabel('Agent Shift')
plt.ylabel('Satisfaction Rate')
plt.show()

print(shift_satisfaction)

##### 1. Why did you pick the specific chart?

Bar chart is suited for comparing satisfaction rate across a small number of categories (5 shifts) — quick visual ranking.

##### 2. What is/are the insight(s) found from the chart?

Split shift has the highest satisfaction (88.1%), while Morning (78.8%) and Night (79.0%) are the lowest — roughly a 9-point gap between best and worst shifts.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — identifies specific shifts (Morning, Night) that may need additional support, training, or staffing review, giving a targeted lever to improve satisfaction without overhauling all operations.

**Any negative-growth insight?** Yes — Night shift likely has fewer staff/supervisors available and possibly lower supervision quality, and issues reported at night may wait longer to be escalated. Since Night is also often when frustrated customers reach out after failed self-service attempts, weak performance here compounds an already higher-friction customer state.

#### Chart - 6

In [ ]:
# Chart 6: Satisfaction Rate by Tenure Bucket

plt.figure(figsize=(7,5))
tenure_satisfaction = data.groupby('Tenure Bucket')['Satisfied'].mean().sort_values()
sns.barplot(x=tenure_satisfaction.index, y=tenure_satisfaction.values, palette='mako')
plt.title('Satisfaction Rate by Agent Tenure Bucket')
plt.xlabel('Tenure Bucket')
plt.ylabel('Satisfaction Rate')
plt.show()

print(tenure_satisfaction)

##### 1. Why did you pick the specific chart?

Bar chart compares satisfaction rate across ordered tenure groups — easy to spot a trend as experience increases.

##### 2. What is/are the insight(s) found from the chart?

satisfaction rises steadily with tenure, from 75.1% (On Job Training) to 83.4% (>90 days). More experienced agents perform better.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — validates that agent training/experience directly improves customer outcomes, supporting investment in longer onboarding, mentorship programs, or reduced ticket load for new agents.

**Any negative-growth insight?** Yes — "On Job Training" agents show the lowest satisfaction (75.1%), meaning new agents are currently handling live customer issues with less proficiency, directly exposing customers to a weaker experience during the agent's learning curve. High agent turnover would keep repeating this weak spot.

#### Chart - 7

In [ ]:
# Chart 7: Top & Bottom Supervisors by Satisfaction Rate

supervisor_satisfaction = data.groupby('Supervisor')['Satisfied'].agg(['mean', 'count'])
supervisor_satisfaction = supervisor_satisfaction[supervisor_satisfaction['count'] >= 30]  # filter for reliability
supervisor_satisfaction = supervisor_satisfaction.sort_values('mean')

plt.figure(figsize=(8,10))
sns.barplot(x=supervisor_satisfaction['mean'].head(10), y=supervisor_satisfaction.head(10).index, palette='Reds_r')
plt.title('Bottom 10 Supervisors by Satisfaction Rate')
plt.xlabel('Satisfaction Rate')
plt.show()

plt.figure(figsize=(8,10))
sns.barplot(x=supervisor_satisfaction['mean'].tail(10), y=supervisor_satisfaction.tail(10).index, palette='Greens')
plt.title('Top 10 Supervisors by Satisfaction Rate')
plt.xlabel('Satisfaction Rate')
plt.show()

# print(f"bottom 10{supervisor_satisfaction.head(10).index}:{supervisor_satisfaction['mean'].head(10)} ")
# print(f"Top 10{supervisor_satisfaction.tail(10).index}:{supervisor_satisfaction['mean'].tail(10)} ")

##### 1. Why did you pick the specific chart?

Split top/bottom bar charts highlight extremes directly — more useful for performance review than showing all 40 supervisors at once.

##### 2. What is/are the insight(s) found from the chart?

bottom supervisor Oliver Nguyen sits at 51.1% satisfaction, while top supervisor Landon Tanaka reaches 89.4% — a ~38-point gap. Most bottom performers are still below 78%, while top performers cluster around 84-89%.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — this pinpoints specific supervisors (e.g., Oliver Nguyen, Austin Johnson) for targeted coaching/review, and top performers (e.g., Landon Tanaka) as potential best-practice benchmarks or mentors for others.

**Any negative-growth insight?** Yes — a 51% satisfaction rate under one supervisor means nearly half their team's customers are dissatisfied. If this supervisor manages a large team long-term without intervention, it's a concentrated, ongoing source of customer churn risk rather than a one-off issue.

#### Chart - 8

In [ ]:
#Chart 8: Sentiment Score Distribution by Satisfaction Status

plt.figure(figsize=(8,6))
sns.boxplot(data=data, x='Satisfied', y='remark_sentiment', palette='coolwarm')
plt.title('Customer Remark Sentiment by Satisfaction Status')
plt.xlabel('Satisfied (0=No, 1=Yes)')
plt.ylabel('Sentiment Score')
plt.show()

print(data.groupby('Satisfied')['remark_sentiment'].mean())

##### 1. Why did you pick the specific chart?

Box plot compares the distribution of a continuous variable (sentiment) across two groups — shows whether the difference is consistent or driven by outliers.

##### 2. What is/are the insight(s) found from the chart?

Dissatisfied customers' remarks average slightly negative (-0.056), while satisfied customers' remarks average positive (0.125) — a clear directional gap, though the average magnitude is modest since 67% of remarks are empty (neutral 0.0), pulling both groups toward zero.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — validates that remark sentiment is a genuine, usable signal for flagging at-risk customers in real time (e.g., auto-flagging negative-sentiment remarks for priority follow-up before the CSAT survey is even submitted).

**Any negative-growth insight?** Yes — since only 33% of customers leave remarks at all, many dissatisfied customers may be silently unhappy (score 1) without ever explaining why in text. This means relying on remarks alone would miss a large share of at-risk customers — the "silent dissatisfied" majority.

#### Chart - 9 - Correlation Heatmap

In [ ]:
# Chart 10: Correlation Heatmap (Numeric Features)

plt.figure(figsize=(10,8))
numeric_cols = ['Satisfied', 'response_time_min_capped', 'response_time_log',
                'remark_sentiment', 'has_remark', 'has_order_info',
                'report_hour', 'report_dayofweek', 'Sub-category_freq', 'Supervisor_freq']

corr = data_encoded[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

# corr = data_encoded[numeric_cols].corr()
# print(corr['Satisfied'].sort_values(ascending=False))

##### 1. Why did you pick the specific chart?

A heatmap/correlation table efficiently summarizes linear relationships between all numeric features and the target at once — useful for spotting the strongest predictors before modeling.

##### 2. What is/are the insight(s) found from the chart?

remark_sentiment has the strongest correlation with satisfaction (+0.25), followed by response_time_log (-0.20) and response_time_min_capped (-0.15) — confirming faster responses and positive sentiment both align with satisfaction. Interestingly, has_remark (-0.096) and has_order_info (-0.12) are both negatively correlated, meaning customers who leave remarks or have order-linked issues tend to be less satisfied. report_hour, report_dayofweek, Supervisor_freq, and Sub-category_freq show negligible correlation (all under 0.04).

#### Chart - 10 - Pair Plot

In [ ]:
# Pair Plot visualization code

pairplot_cols = ['Satisfied', 'response_time_min_capped', 'remark_sentiment', 'report_hour']

sns.pairplot(data_encoded[pairplot_cols], hue='Satisfied', palette='coolwarm', diag_kind='kde', plot_kws={'alpha':0.4, 's':15})
plt.suptitle('Pairplot of Key Numeric Features by Satisfaction', y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Answer Here.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis (H₀)**: There is no significant difference in the mean response time between Satisfied and Not Satisfied customers. (μ_satisfied = μ_not_satisfied)

**Alternate Hypothesis (H₁)**: There is a significant difference in the mean response time between Satisfied and Not Satisfied customers. (μ_satisfied ≠ μ_not_satisfied)

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy import stats

# H0: No difference in mean response time between Satisfied and Not Satisfied
# H1: There IS a difference in mean response time between the two groups

satisfied_time = data_encoded[data_encoded['Satisfied']==1]['response_time_min_capped']
not_satisfied_time = data_encoded[data_encoded['Satisfied']==0]['response_time_min_capped']

t_stat1, p_value1 = stats.ttest_ind(satisfied_time, not_satisfied_time, equal_var=False)

print("Hypothesis 1: Response Time vs Satisfaction")
print("T-statistic:", t_stat1)
print("P-value:", p_value1)

alpha = 0.05
if p_value1 < alpha:
    print("Result: Reject H0 -> Response time significantly differs between groups.\n")
else:
    print("Result: Fail to reject H0 -> No significant difference found.\n")


##### Which statistical test have you done to obtain P-Value?

Independent samples t-test (Welch's t-test)

##### Why did you choose the specific statistical test?

We're comparing the mean of a continuous variable (response_time_min_capped) between two independent groups (Satisfied vs Not Satisfied). A t-test is the standard choice for comparing means between two groups. We specifically use Welch's version (equal_var=False) rather than the standard t-test because the two groups likely have unequal variances (dissatisfied customers showed much wider spread in response time — mean 296 vs 126), and Welch's t-test doesn't assume equal variance, making it more robust here.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# H0: Satisfaction is independent of channel
# H1: Satisfaction is dependent on channel

contingency_table = pd.crosstab(data['channel_name'], data['Satisfied'])
print(contingency_table)

chi2_stat, p_value2, dof, expected = stats.chi2_contingency(contingency_table)

print("\nHypothesis 2: Channel vs Satisfaction")
print("Chi2 Statistic:", chi2_stat)
print("P-value:", p_value2)
print("Degrees of Freedom:", dof)

if p_value2 < 0.05:
    print("Result: Reject H0 -> Satisfaction depends on channel.")
else:
    print("Result: Fail to reject H0 -> No significant association found.")

##### Which statistical test have you done to obtain P-Value?

Chi-Square Test of Independence

##### Why did you choose the specific statistical test?

Both variables here are categorical — channel_name (Email/Inbound/Outcall) and Satisfied (0/1). The Chi-Square test is specifically designed to check whether two categorical variables are statistically independent or associated, by comparing observed frequencies (actual counts in each channel/satisfaction combination) against expected frequencies (what we'd expect if there were no relationship at all).

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# H0: No difference in mean sentiment between Satisfied and Not Satisfied
# H1: There IS a difference in mean sentiment between the two groups

satisfied_sentiment = data_encoded[data_encoded['Satisfied']==1]['remark_sentiment']
not_satisfied_sentiment = data_encoded[data_encoded['Satisfied']==0]['remark_sentiment']

t_stat3, p_value3 = stats.ttest_ind(satisfied_sentiment, not_satisfied_sentiment, equal_var=False)

print("Hypothesis 3: Remark Sentiment vs Satisfaction")
print("T-statistic:", t_stat3)
print("P-value:", p_value3)

if p_value3 < alpha:
    print("Result: Reject H0 -> Sentiment significantly differs between groups.\n")
else:
    print("Result: Fail to reject H0 -> No significant difference found.\n")

##### Which statistical test have you done to obtain P-Value?

independent samples t-test (Welch's t-test)

##### Why did you choose the specific statistical test?

Same structure as Hypothesis 1 — comparing the mean of a continuous variable (remark_sentiment) between two independent groups (Satisfied vs Not Satisfied). We use Welch's t-test again rather than assuming equal variances between groups, since we have no prior reason to assume the sentiment score variance is identical across both groups, and Welch's is the safer default when that assumption hasn't been verified.













     






















            

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation

#### What all missing value imputation techniques have you used and why did you use those techniques?

Column dropping — for columns with extreme missingness (>80%): connected_handling_time, order_date_time, Customer_City, Item_price, Product_category
Missing indicator/flag creation — has_order_info and has_remark flags, to preserve the signal of missingness itself
Constant/placeholder imputation — filled missing Customer Remarks with empty string ''
Median imputation — for invalid/negative response_time_min values
Row deletion (listwise deletion) — dropped the single row with cascading nulls that couldn't be reasonably imputed

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments

##### What all outlier treatment techniques have you used and why did you use those techniques?

Technique used: Percentile Capping (Winsorization) at the 99th percentile, applied to response_time_min.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns

#### What all categorical encoding techniques have you used & why did you use those techniques?

One-Hot Encoding (with drop_first=True) — for low-cardinality columns: channel_name, category, Tenure Bucket, Agent Shift

Frequency Encoding — for medium-cardinality columns: Sub-category, Supervisor

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction

#### 2. Lower Casing

In [ ]:
# Lower Casing

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords

In [ ]:
# Remove White spaces

#### 6. Rephrase Text

In [ ]:
# Rephrase Text

#### 7. Tokenization

In [ ]:
# Tokenization

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)

##### Which text normalization technique have you used and why?

Answer Here.

#### 9. Part of speech tagging

In [ ]:
# POS Taging

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text

##### Which text vectorization technique have you used and why?

Answer Here.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting

##### What all feature selection methods have you used  and why?

Answer Here.

##### Which all features you found important and why?

Answer Here.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data

### 6. Data Scaling

In [ ]:
# Scaling your data

##### Which method have you used to scale you data and why?

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Answer Here.

In [ ]:
# DImensionality Reduction (If needed)

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Answer Here.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.

##### What data splitting ratio have you used and why?

Answer Here.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Answer Here.

In [ ]:
# Handling Imbalanced Dataset (If needed)

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=['Satisfied'])
y = model_df['Satisfied']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

In [ ]:
# ML Model - 1 Implementation

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Logistic Regression is scale-sensitive, so we scale features first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                               f1_score, roc_auc_score, confusion_matrix,
                               classification_report, RocCurveDisplay)

print("=== Logistic Regression Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Satisfied','Satisfied'],
            yticklabels=['Not Satisfied','Satisfied'])
plt.title('Confusion Matrix - Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob_lr)
plt.title('ROC Curve - Logistic Regression')
plt.show()

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

from sklearn.model_selection import StratifiedKFold, GridSearchCV

# --- Stratified K-Fold: preserves class ratio in each fold, critical given our imbalance ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Hyperparameter grid for Logistic Regression ---
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],        # inverse of regularization strength
    'penalty': ['l1', 'l2'],              # type of regularization
    'solver': ['liblinear']                # solver that supports both l1 and l2
}

grid_lr = GridSearchCV(
    estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    param_grid=param_grid_lr,
    scoring='f1',                          # optimize F1, not accuracy, given imbalance
    cv=cv,
    n_jobs=-1
)

grid_lr.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_lr.best_params_)
print("Best CV F1 Score:", grid_lr.best_score_)

# --- Evaluate tuned model on test set ---
best_lr = grid_lr.best_estimator_
y_pred_lr_tuned = best_lr.predict(X_test_scaled)
y_prob_lr_tuned = best_lr.predict_proba(X_test_scaled)[:, 1]

print("\n=== Tuned Logistic Regression Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr_tuned))
print("Precision:", precision_score(y_test, y_pred_lr_tuned))
print("Recall:", recall_score(y_test, y_pred_lr_tuned))
print("F1 Score:", f1_score(y_test, y_pred_lr_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr_tuned))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr_tuned))

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Decision tree

from sklearn.tree import DecisionTreeClassifier

# Decision Trees don't need feature scaling (splits are based on thresholds, not magnitude)
dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print("=== Decision Tree Performance (Default) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("F1 Score:", f1_score(y_test, y_pred_dt))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_dt))
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

param_grid_dt = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

grid_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(class_weight='balanced', random_state=42),
    param_grid=param_grid_dt,
    scoring='f1',
    cv=cv,
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("Best Parameters:", grid_dt.best_params_)
print("Best CV F1 Score:", grid_dt.best_score_)

best_dt = grid_dt.best_estimator_
y_pred_dt_tuned = best_dt.predict(X_test)
y_prob_dt_tuned = best_dt.predict_proba(X_test)[:, 1]

print("\n=== Tuned Decision Tree Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred_dt_tuned))
print("Precision:", precision_score(y_test, y_pred_dt_tuned))
print("Recall:", recall_score(y_test, y_pred_dt_tuned))
print("F1 Score:", f1_score(y_test, y_pred_dt_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_dt_tuned))
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt_tuned))

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Answer Here.

### ML Model - 3

In [ ]:
# ML Model - 3 Implementation

from sklearn.ensemble import RandomForestClassifier

# Default Random Forest first
rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest Performance (Default) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

# Using RandomizedSearchCV instead of GridSearchCV — full grid here would be 3*4*3*3=108 combos * 5 folds = 540 fits, too slow
from sklearn.model_selection import RandomizedSearchCV

random_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=param_grid_rf,
    n_iter=20,               # test 20 random combinations instead of all 108
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1
)

random_rf.fit(X_train, y_train)

print("Best Parameters:", random_rf.best_params_)
print("Best CV F1 Score:", random_rf.best_score_)

best_rf = random_rf.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test)
y_prob_rf_tuned = best_rf.predict_proba(X_test)[:, 1]

print("\n=== Tuned Random Forest Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf_tuned))
print("Precision:", precision_score(y_test, y_pred_rf_tuned))
print("Recall:", recall_score(y_test, y_pred_rf_tuned))
print("F1 Score:", f1_score(y_test, y_pred_rf_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf_tuned))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf_tuned))

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

In [ ]:
# Consolidate all tuned model results
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [0.7120, 0.7654, 0.8440],
    'Precision': [0.9030, 0.8602, 0.8560],
    'Recall': [0.7290, 0.8544, 0.9748],
    'F1 Score': [0.8067, 0.8573, 0.9116],
    'ROC-AUC': [0.7586, 0.6010, 0.7398]
})

print(results)

# Melt for grouped bar chart
results_melted = results.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(12,6))
sns.barplot(data=results_melted, x='Metric', y='Score', hue='Model', palette='mako')
plt.title('Model Performance Comparison Across Evaluation Metrics')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.show()

In [ ]:
# ROC Curves for all 3 models on one plot for direct visual comparison
from sklearn.metrics import roc_curve

plt.figure(figsize=(8,6))

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr_tuned)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt_tuned)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf_tuned)

plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={roc_auc_score(y_test, y_prob_lr_tuned):.3f})')
plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC={roc_auc_score(y_test, y_prob_dt_tuned):.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={roc_auc_score(y_test, y_prob_rf_tuned):.3f})')
plt.plot([0,1],[0,1],'k--', label='Random Guess')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

Answer Here.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

 **Final Model Chosen: Logistic Regression**

Despite Random Forest's higher accuracy (84.4%) and F1 (0.912), it only catches 23% of dissatisfied customers (recall). Logistic Regression catches 73% of dissatisfied customers, has the best ROC-AUC (0.759), and is more interpretable — making it better suited to the business goal of proactively identifying at-risk customers, not just maximizing overall accuracy.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Coefficients
coefficients = pd.Series(best_lr.coef_[0], index=X.columns).sort_values(ascending=False)
print(coefficients)

# SHAP values
import shap
explainer = shap.LinearExplainer(best_lr, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)
shap.summary_plot(shap_values, X_test, feature_names=X.columns, plot_type='bar')

Our final model, Logistic Regression, is inherently interpretable — each feature's coefficient directly shows its effect on the log-odds of being "Satisfied" (positive = increases satisfaction likelihood, negative = decreases it).

For a more robust view, we also used SHAP (LinearExplainer) — chosen because it's fast and exact for linear models — to see each feature's actual average impact on predictions, accounting for how the feature varies in real data (not just its raw coefficient).

Both methods confirm response_time_log, remark_sentiment, and has_order_info as the strongest drivers of satisfaction — aligning with our earlier EDA findings.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project set out to identify the key drivers of customer satisfaction at Flipkart and build a model to predict at-risk (dissatisfied) customers, using ~22,430 customer service interactions across channels, categories, and support teams.

**Key Findings:**

CSAT is heavily polarized — 68.5% of customers are highly satisfied (score 5) while 14% are highly dissatisfied (score 1), with very few "neutral" ratings
Response time is one of the strongest satisfaction drivers — dissatisfied customers waited 4x longer (20 min median) than satisfied ones (5 min median)
Remark sentiment correlates meaningfully with satisfaction, though only 33% of customers leave feedback text — meaning many dissatisfied customers stay "silent"
Email channel underperforms (70.5% satisfaction) compared to Inbound/Outcall (~81.5%)
Order Related and Cancellation categories show the lowest satisfaction, pointing to core transactional friction
Agent tenure and supervisor both show meaningful performance variation — newer agents and specific supervisors correlate with lower satisfaction, highlighting training/coaching opportunities
Hypothesis testing statistically confirmed response time, channel, and sentiment all significantly relate to satisfaction

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***